# Cluster cohorts → Bray-Curtis (day 0 → last day)

Two parts:

1. **Recreate the cohort pipeline** (the lost code): build the genus abundance matrix + metadata,
   attach a `day` column (from `canon.xlsx`) and the cluster assignment (from the Master Map),
   then split into the four `cluster_*_cohort` folders under `Notebooks/out/`.
2. **Bray-Curtis per subject**: for each subject, dissimilarity between its **day 0** sample and its
   **last day** sample (on relative abundance), then one figure with **4 box plots** — one per cluster.

> ⚠️ **Builder bug (this branch).** `AbundanceMatrixBuilder.build(matrix, 'genus')` on
> `refactor-abundance-matrix-two-state` mis-aggregates counts: row sums come out ~385× too large and
> every sample collapses to a near-identical, wrong profile (Prevotella ≈0.5 everywhere), which flattens
> Bray-Curtis to noise (cross-subject BC ≈0.04). We therefore build the genus matrix **directly from the
> `self_count` of genus-rank rows** — the documented semantics the builder is meant to implement. With
> the correct matrix, cross-subject BC ≈0.5 (realistic). The pre-existing cohort `abundance_matrix.csv`
> files were built with the broken builder and are regenerated here.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import braycurtis

from askslm import metadata, taxonomy

ROOT = Path("/Users/yarintamam/Code/Ask-SLM")
INPUTS = ROOT / "Workspaces/ABX/inputs"
CANON = INPUTS / "Supplementary/canon.xlsx"
MASTER_MAP = ROOT / "Notebooks/Viral_Ecological_Clusters_Master_Map-v2.xlsx"
OUT = ROOT / "Notebooks/out"
BC_DIR = OUT / "braycurtis"

## 1a. Metadata: extract → add `day` → add cluster

* `day` comes from `canon.xlsx`, joined on `sample_name == library`.
* `Cluster_ID` / `Cluster_Name` come from the Master Map, joined on `host_subject_id_sam == Subject`.
* Cohorts are the **WGS** samples of the clustered subjects (matches the original cohort sizes).

In [ ]:
metadata_df = metadata.extract(str(INPUTS / "metadata.csv"))

canon = pd.read_excel(CANON)[["library", "day"]]
cluster_map = pd.read_excel(MASTER_MAP)  # Subject, Cluster_ID, Cluster_Name

metadata_df = (
    metadata_df
    .merge(cluster_map, left_on="host_subject_id_sam", right_on="Subject", how="left")
    .merge(canon, left_on="sample_name", right_on="library", how="left")
    .drop(columns=["Subject", "library"])
)

cohort_md = metadata_df[
    metadata_df["Cluster_ID"].notna() & (metadata_df["assay_type"] == "WGS")
].copy()
cohort_md["Cluster_ID"] = cohort_md["Cluster_ID"].astype(int)

print("cohort samples:", len(cohort_md), "| day non-null:", cohort_md["day"].notna().sum())
print(cohort_md.groupby("Cluster_ID").size().to_dict())

## 1b. Genus abundance matrix — directly from `self_count`

Pivot the genus-rank rows of `taxonomy.csv`: `acc` × genus `name`, values = `self_count`
(direct counts, never the cumulative `total_count`). This is the builder's intended behaviour,
computed here without the broken collapse — see the bug note at the top.

In [ ]:
taxonomy_df = taxonomy.load(str(INPUTS / "taxonomy.csv"))
genus_rows = taxonomy_df[taxonomy_df["rank"] == "genus"]
genus_matrix = genus_rows.pivot_table(
    index="acc", columns="name", values="self_count", aggfunc="sum", fill_value=0
)
print("genus matrix (samples x genera):", genus_matrix.shape)
# sanity: cross-subject BC should be realistic (~0.5), not ~0.04
a, b = genus_matrix.iloc[0], genus_matrix.iloc[100]
print("spot-check cross-sample BC:", round(braycurtis((a / a.sum()).values, (b / b.sum()).values), 3))

## 2. Split into the four cluster cohorts and write them out

Each `cluster_<i>_cohort/` gets `metadata.csv` (now including the `day` column) and a tab-delimited
`abundance_matrix.csv` (acc × genus). Columns are genus names (the correct `self_count` matrix).

In [ ]:
cohorts = {}
for cid, md in cohort_md.groupby("Cluster_ID"):
    accs = [a for a in md["acc"] if a in genus_matrix.index]
    sub_matrix = genus_matrix.loc[accs]
    sub_matrix = sub_matrix.loc[:, sub_matrix.sum(axis=0) > 0]  # drop genera absent from this cohort
    md = md[md["acc"].isin(accs)]
    cohorts[cid] = (md, sub_matrix)

    out_dir = OUT / f"cluster_{cid}_cohort"
    out_dir.mkdir(parents=True, exist_ok=True)
    md.to_csv(out_dir / "metadata.csv", index=False)
    sub_matrix.to_csv(out_dir / "abundance_matrix.csv", sep="\t", index_label="acc")
    print(f"cluster_{cid}: {len(md)} samples, {md['host_subject_id_sam'].nunique()} subjects, "
          f"{sub_matrix.shape[1]} genera -> {out_dir.name}")

## 3. Bray-Curtis: day 0 → last day, per subject

Per subject:
* **day 0** = the sample with `day == 0`; if a subject has no exact day-0 sample (BAC, EAO),
  fall back to the day closest to 0 and record the day actually used.
* **last day** = the sample with the largest `day`.
* Both count vectors → **relative abundance**, then Bray-Curtis between them.

In [ ]:
def relative_abundance(counts: pd.Series) -> np.ndarray:
    total = counts.sum()
    return (counts / total).to_numpy() if total else counts.to_numpy(dtype=float)

def baseline_and_last(subject_md: pd.DataFrame):
    """Return (day0_row, last_row); baseline is day==0 or, failing that, the day closest to 0."""
    s = subject_md.dropna(subset=["day"]).sort_values("day")
    exact = s[s["day"] == 0]
    day0 = exact.iloc[0] if len(exact) else s.iloc[s["day"].abs().argmin()]
    return day0, s.iloc[-1]

records = {}
for cid, (md, sub_matrix) in cohorts.items():
    cluster_name = md["Cluster_Name"].iloc[0]
    rows = []
    for subject, sub_md in md.groupby("host_subject_id_sam"):
        if sub_md["day"].dropna().nunique() < 2:
            continue  # need two distinct timepoints
        day0, last = baseline_and_last(sub_md)
        v0 = relative_abundance(sub_matrix.loc[day0["acc"]])
        vl = relative_abundance(sub_matrix.loc[last["acc"]])
        rows.append({
            "subject": subject, "cluster_id": cid, "cluster_name": cluster_name,
            "day0_acc": day0["acc"], "day0_day": int(day0["day"]),
            "lastday_acc": last["acc"], "lastday_day": int(last["day"]),
            "braycurtis": braycurtis(v0, vl),
        })
    records[cid] = pd.DataFrame(rows).sort_values("subject").reset_index(drop=True)
    print(f"cluster_{cid}: {len(records[cid])} subjects, BC mean={records[cid]['braycurtis'].mean():.3f}")
records[0].head()

## 4. Save per-cluster Bray-Curtis files

In [ ]:
BC_DIR.mkdir(parents=True, exist_ok=True)
for cid, df in records.items():
    df.to_csv(BC_DIR / f"cluster_{cid}_braycurtis.csv", index=False)
    print("wrote", BC_DIR / f"cluster_{cid}_braycurtis.csv")

combined = pd.concat(records.values(), ignore_index=True)
combined.to_csv(BC_DIR / "all_clusters_braycurtis.csv", index=False)
print("wrote", BC_DIR / "all_clusters_braycurtis.csv")

## 5. One figure, four box plots (one per cluster)

In [ ]:
order = sorted(records)
data = [records[cid]["braycurtis"].values for cid in order]

def short(name: str) -> str:
    head, _, tail = name.partition(":")
    label = tail.split("(")[0].strip() if tail else ""
    return f"{head.strip()}\n{label}"

labels = [short(records[cid]["cluster_name"].iloc[0]) for cid in order]
colors = [f"C{i}" for i in range(len(order))]  # matplotlib default palette
EDGE = "0.4"  # neutral gray for all line elements

fig, ax = plt.subplots(figsize=(9, 6), facecolor="white")
ax.set_facecolor("white")

bp = ax.boxplot(
    data, tick_labels=labels, patch_artist=True, showmeans=True, widths=0.55,
    boxprops=dict(linewidth=1.2, edgecolor=EDGE),
    medianprops=dict(color=EDGE, linewidth=1.4),
    whiskerprops=dict(color=EDGE, linewidth=1.2),
    capprops=dict(color=EDGE, linewidth=1.2),
    flierprops=dict(marker="o", markersize=4, markerfacecolor="none", markeredgecolor="0.6"),
    meanprops=dict(marker="D", markerfacecolor="white", markeredgecolor=EDGE, markersize=6),
)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color); patch.set_alpha(0.55)
for i, vals in enumerate(data, start=1):
    ax.scatter(np.random.normal(i, 0.05, len(vals)), vals, s=18, color="0.45", alpha=0.7, zorder=3)

ax.set_ylabel("Bray-Curtis dissimilarity (day 0 → last day)")
ax.set_title("Within-subject community shift by cluster")
ax.set_ylim(0, 1)
ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.grid(True, linestyle="--", linewidth=0.6, alpha=0.4)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(BC_DIR / "braycurtis_boxplots.png", dpi=200, facecolor="white", bbox_inches="tight")
print("wrote", BC_DIR / "braycurtis_boxplots.png")
plt.show()